In [1]:
# ----------------------------
# Bank System — Data Driven Demo (Unified)
# ----------------------------

import sys
sys.path.append("../src")

from datetime import datetime, date, timedelta
from collections import Counter, defaultdict

from bank.bank import Bank
from bank.client import Client, Contact

from account.bank_account import BankAccount
from account.savings_account import SavingsAccount
from account.premium_account import PremiumAccount

from account.enums import AccountCurrency

from transaction.transaction import Transaction
from transaction.transaction_queue import TransactionAlreadyInQueue, QueueIsEmpty


# ----------------------------
# RAW DATA
# ----------------------------

DATA = {
    "clients": [
        {"id": "C001", "name": "Ivan Petrov"},
        {"id": "C002", "name": "Anna Smirnova"},
        {"id": "C003", "name": "John Smith"},
        {"id": "C004", "name": "Maria Garcia"},
        {"id": "C005", "name": "Liam Brown"},
        {"id": "C006", "name": "Emma Wilson"},
        {"id": "C007", "name": "Noah Martinez"},
    ],

    "accounts": [
        {"id": "A001", "client_id": "C001", "type": "debit", "balance": 5000, "currency": "USD"},
        {"id": "A002", "client_id": "C001", "type": "savings", "balance": 2000, "currency": "EUR"},
        {"id": "A003", "client_id": "C002", "type": "debit", "balance": 8000, "currency": "USD"},
        {"id": "A004", "client_id": "C002", "type": "debit", "balance": 1500, "currency": "EUR"},
        {"id": "A005", "client_id": "C003", "type": "premium", "balance": 12000, "currency": "USD"},
        {"id": "A006", "client_id": "C004", "type": "debit", "balance": 3000, "currency": "EUR"},
        {"id": "A007", "client_id": "C004", "type": "savings", "balance": 700, "currency": "USD"},
        {"id": "A008", "client_id": "C005", "type": "debit", "balance": 4000, "currency": "USD"},
        {"id": "A009", "client_id": "C006", "type": "premium", "balance": 9500, "currency": "EUR"},
        {"id": "A010", "client_id": "C007", "type": "debit", "balance": 1000, "currency": "USD"},
    ],

    "transactions": [
        ("A001","A003",500,"USD"),
        ("A002","A006",300,"EUR"),
        ("A003","A005",7000,"USD"),
        ("A004","A006",400,"EUR"),
        ("A006","A007",500,"EUR"),
        ("A007","A008",600,"USD"),
        ("A008","A009",3500,"USD"),
        ("A009","A002",1000,"EUR"),
        ("A010","A003",200,"USD"),
        ("A002","A003",1500,"EUR"),
        ("A003","A004",9000,"USD"),
        ("A005","A001",10000,"USD"),
        ("A005","A009",11000,"USD"),
        ("A009","A003",8000,"EUR"),

        # frequent pattern
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),
        ("A001","A003",200,"USD"),

        # more normal flow
        ("A002","A004",250,"EUR"),
        ("A006","A002",300,"EUR"),
        ("A007","A001",150,"USD"),
        ("A008","A007",400,"USD"),
        ("A001","A010",350,"USD"),
        ("A003","A006",1200,"USD"),
        ("A004","A005",500,"EUR"),
        ("A005","A006",700,"USD"),

        # error cases
        ("A010","A003",999999,"USD"), # limit exceeded

        # 🔥 HIGH RISK — большие суммы
        ("A001","A003",15000,"USD"),
        ("A002","A005",18000,"EUR"),
        ("A003","A009",20000,"USD"),
        ("A005","A001",12000,"USD"),
        ("A009","A003",14000,"EUR"),

        # 🧊 ЗАМОРОЖЕННЫЙ СЧЕТ (сергей = A010)
        ("A001","A010",300,"USD"),
        ("A002","A010",700,"EUR"),
        ("A005","A010",1500,"USD"),

        # 🔥 подозрительные (разные источники → один получатель)
        ("A001","A003",3000,"USD"),
        ("A002","A003",3000,"USD"),
        ("A005","A003",3000,"USD"),

        # 🔥 очень большие + повтор
        ("A005","A003",25000,"USD"),  # limit breach
        ("A005","A003",25000,"USD"),

        # ❌ попытка использовать уже замороженный сценарий
        ("A010","A001",500,"USD"),
    ]
}


# ----------------------------
# INIT BANK
# ----------------------------

bank = Bank()
print("Bank created")


# ----------------------------
# CLIENTS
# ----------------------------

clients = {}

for c in DATA["clients"]:
    client = Client(
        name=c["name"],
        client_id=c["id"],
        date_of_birth=date(1990, 1, 1),
        login=c["id"],
        password="test",
        contacts=[Contact("email", f"{c['id']}@test.com")]
    )
    bank.add_client(client)
    clients[c["id"]] = client

print(f"Clients loaded: {len(clients)}")


# ----------------------------
# ACCOUNTS
# ----------------------------

accounts = {}

def create_account(acc):
    if acc["type"] == "debit":
        return BankAccount(owner_name=clients[acc["client_id"]].name)
    elif acc["type"] == "savings":
        return SavingsAccount(owner_name=clients[acc["client_id"]].name, min_balance=100, monthly_interest=1)
    elif acc["type"] == "premium":
        return PremiumAccount(owner_name=clients[acc["client_id"]].name, fee=5)

for acc in DATA["accounts"]:
    account = create_account(acc)

    bank.open_account(acc["client_id"], account)

    # initial balance
    account.deposit(acc["balance"])

    accounts[acc["id"]] = account

print(f"Accounts loaded: {len(accounts)}")


# ----------------------------
# TRANSACTIONS
# ----------------------------

queue = bank.transaction_queue
transactions = []

for from_id, to_id, amount, currency in DATA["transactions"]:
    to_acc = accounts.get(to_id)

    if not to_acc:
        continue

    tx = Transaction(
        amount,
        AccountCurrency[currency],
        1,
        accounts[from_id],
        to_acc
    )

    transactions.append(tx)

    try:
        queue.enqueue(tx)
    except TransactionAlreadyInQueue:
        pass


# ----------------------------
# ADD SUSPICIOUS LOAD
# ----------------------------

for _ in range(6):
    queue.enqueue(
        Transaction(
            200,
            AccountCurrency.USD,
            1,
            accounts["A001"],
            accounts["A003"]
        )
    )

# freeze account scenario
bank.freeze_account(accounts["A010"].id)

print(f"Transactions prepared: {len(transactions)}")


# ----------------------------
# PROCESS
# ----------------------------

processor = bank.transaction_processor

while True:
    try:
        processor.process()
    except QueueIsEmpty:
        break


# ----------------------------
# LOGGING
# ----------------------------

print("\n=== RESULTS ===")

for t in transactions:
    print(
        t.transaction_id,
        t.status.name,
        "| reject:",
        t.reject_reason
    )


# ----------------------------
# REPORTS
# ----------------------------

print("\nSuspicious:")

for log in bank.audit_log.get_suspicious_transactions():
    print(
        getattr(log, "client_id", "-"),
        getattr(log, "risk", "-")
    )


print("\nClient risk profiles:")

for cid in clients:
    profile = bank.audit_log.get_client_risk_profile(cid)
    print(cid, dict(profile))


print("\nErrors:")

for err in bank.audit_log.get_errors():
    print(err)


# ----------------------------
# STATISTICS
# ----------------------------

print("\nStatistics:")

stats = {
    "total": len(transactions),
    "success": sum(1 for t in transactions if t.status.name == "SUCCESS"),
    "failed": sum(1 for t in transactions if t.status.name == "FAILED"),
    "rejected": sum(1 for t in transactions if t.reject_reason),
}

print(stats)


# ----------------------------
# TOTAL BALANCE
# ----------------------------

total_balance = sum(acc.balance for acc in accounts.values())

print("\nTotal balance:", total_balance)

Bank created
Clients loaded: 7
Accounts loaded: 10
Transactions prepared: 49
total risk: 25 | transaction : 1a6bf328-d54a-4c58-9c52-7192f159dbe1
total risk: 28 | transaction : 2dea4b0a-298c-4e25-ae88-766157c59a05
total risk: 95 | transaction : 1aea3a40-1122-429e-9270-5084ada6340f
total risk: 34 | transaction : adc6e29d-3990-4bfa-87b2-2223093ac24d
total risk: 35 | transaction : 599effe6-ed07-4ef1-8ebd-2bc4517feb7a
total risk: 46 | transaction : 1aa0319c-cda9-4648-88d7-2ae6b7412bcd
total risk: 60 | transaction : 36abc56c-f846-4af5-9274-13cbe014c518
total risk: 30 | transaction : bf1e2c7b-5dc2-426c-9e60-0e6c40850bf5
total risk: 22 | transaction : 5e00e7d2-d283-499e-95cc-440ff0ee6274
total risk: 50 | transaction : df922482-5683-47f7-9339-6aeebdd31344
total risk: 130 | transaction : b3a64256-6202-49ce-9475-40a718607cd1
total risk: 120 | transaction : 42b22de6-b8c0-4ff3-bcd3-9e0fdb3602c5
total risk: 135 | transaction : a259e3cc-6c0a-49a6-bb1d-c840ac485fde
total risk: 105 | transaction : cc36